# Pencarian dan pemeriksaan koordinat puskesmas yang hilang — v4

Jalankan **Run All** dengan kernel `.venv` proyek, dari folder proyek atau `Archives`.
Notebook menggunakan master fasilitas, poligon kecamatan, snapshot Healthsites, dan bukti riwayat OSM yang sudah tersedia di proyek. Tidak memerlukan password Supabase dan tidak mengubah database atau CSV asli.

**Pemeriksaan otomatis bukan survei lapangan.** Nama tepat + batas wilayah + riwayat sebelum akhir 2024 menghasilkan kandidat historis, bukan persetujuan publikasi. Alamat diperiksa sebagai bukti tambahan; tidak ada fuzzy matching Manding/Ganding, fallback jalan/desa/kecamatan, atau pengisian angka tebakan.

Default memakai arsip lokal. Aktifkan `ALLOW_NETWORK` untuk melengkapi riwayat objek/node yang belum diarsipkan. Unduhan berurutan, dibatasi, disimpan dengan SHA-256, dan dapat dilanjutkan. Riwayat mengikuti [OSM API v0.6](https://wiki.openstreetmap.org/wiki/API_v0.6#History). Data OSM © OpenStreetMap contributors, ODbL; periksa juga ketentuan sumber Healthsites.

Ini notebook proyek lokal, bukan file Colab mandiri: folder `data` dan modul `python/gis_pipeline` harus ikut tersedia. Dependensi: `shapely`, `pandas`; sudah tersedia di `.venv` proyek.

In [1]:
import json, re, hashlib, sys, os, math
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone
import pandas as pd
from shapely.geometry import shape, Point, Polygon

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/facility-master.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari checkout proyek dengan data/processed tersedia.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'python'))
from gis_pipeline.etl.http_client import SourceClient, atomic_json
ALLOW_NETWORK = False
MAX_FETCHES = 250
CUTOFF = '2024-12-31T23:59:59Z'
OUT = ROOT / 'data/exports/coordinate-review-v4'
OUT.mkdir(parents=True, exist_ok=True)
client = SourceClient('data/raw', ['api.openstreetmap.org'], interval=1.1, deadline=40)
inputs = {}
def read(path):
    path = Path(path)
    raw = path.read_bytes()
    inputs[path.as_posix()] = hashlib.sha256(raw).hexdigest()
    return json.loads(raw.decode('utf-8-sig'))
def receipt_data(receipt):
    path = Path(receipt['raw_path'].replace('\\', '/'))
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != receipt['sha256']:
        raise ValueError('SHA-256 bukti tidak cocok: ' + str(path))
    inputs[path.as_posix()] = receipt['sha256']
    return json.loads(raw)
master = read('data/processed/facility-master.json')['facilities']
assert len(master) == len({m['facility_code'] for m in master})
boundaries = {f['properties']['region_code']: shape(f['geometry'])
              for f in read('data/processed/boundaries/analysis.geojson')['features']}
assert all(g.is_valid and not g.is_empty for g in boundaries.values())
features = receipt_data(read('docs/pekerjaan/WF-068/healthsites-points-receipt.json'))['features']
# Keputusan yang sudah tercatat dipertahankan; bukan membaca status database langsung.
accepted = set()
for path in sorted(Path('docs/pekerjaan/WF-068').glob('*live-review.json')):
    decision = read(path)
    if decision.get('decision') == 'accepted' and decision.get('facility_code'):
        accepted.add(str(decision['facility_code']))
targets = [m for m in master if not m['baseline_coordinate_present'] and m['facility_code'] not in accepted]
print('Target baseline hilang, dikurangi keputusan diterima yang tercatat:', len(targets))
print('Tidak menyatakan snapshot database terkini; keputusan lokal:', sorted(accepted))

Target baseline hilang, dikurangi keputusan diterima yang tercatat: 407
Tidak menyatakan snapshot database terkini; keputusan lokal: ['35180200006', '35250200032', '35730200010']


## 1. Pencarian kandidat
Nama dinormalisasi hanya untuk awalan fasilitas dan tanda baca. Wilayah dicocokkan dengan **kode kecamatan dan geometri**, bukan substring nama kota/kabupaten. Tidak ada bounding box lintang yang memotong Bawean. Titik representatif poligon hanya untuk pencarian, belum menjadi koordinat fasilitas.

In [2]:
def normalized(name):
    text = re.sub(r'\b(puskesmas|uptd|upt)\b', '', str(name).casefold())
    return re.sub(r'[^a-z0-9]', '', text)
def excluded_name(name):
    return bool(set(re.findall(r'\w+', str(name).casefold())) & {'pustu', 'pembantu', 'keliling'})
index = defaultdict(list)
for feature in features:
    name = feature['properties'].get('name') or ''
    if normalized(name) and not excluded_name(name):
        index[normalized(name)].append(feature)
discovery = []
for facility in targets:
    boundary = boundaries[facility['region_code']]
    found = {}
    for feature in index[normalized(facility['attributes']['nama_puskesmas'])]:
        geom = shape(feature['geometry'])
        if geom.is_empty or not geom.is_valid or not boundary.covers(geom.representative_point()):
            continue
        props = feature['properties']
        obj = str(props['osm_type']) + '/' + str(props['osm_id'])
        found[obj] = feature
    discovery.append((facility, found))
print('Fasilitas dengan kandidat:', sum(bool(c) for _, c in discovery))
print('Fasilitas tanpa kandidat dalam snapshot ini:', sum(not c for _, c in discovery))

Fasilitas dengan kandidat: 40
Fasilitas tanpa kandidat dalam snapshot ini: 367


## 2. Pemeriksaan riwayat dan geometri
Versi objek dan setiap node poligon dipilih pada/before cutoff, termasuk status penghapusan. Poligon harus tertutup dan valid. Titik dalam poligon diberi label `polygon_representative`, bukan pintu masuk/GPS. Tidak adanya rekaman 2024 berarti bukti historis belum ada, bukan bukti fasilitas belum berdiri.

In [3]:
receipts = {}
for record in read('data/staging/missing-candidate-histories.json')['records']:
    if record.get('receipt'):
        receipts[record['osm_object']] = record['receipt']
saved_path = OUT / 'history-receipts.json'
if saved_path.exists():
    receipts.update(read(saved_path))
fetches = 0
memo = {}
def historical(obj):
    global fetches
    if obj in memo:
        return memo[obj]
    if not re.fullmatch(r'(node|way)/[0-9]+', obj):
        raise ValueError('unsupported_object')
    if obj in receipts:
        doc = receipt_data(receipts[obj])
    else:
        if not ALLOW_NETWORK:
            raise ValueError('history_not_cached')
        if fetches >= MAX_FETCHES:
            raise ValueError('fetch_budget_reached')
        fetches += 1
        result = client.fetch('https://api.openstreetmap.org/api/0.6/' + obj + '/history.json',
                              'coordinate-v4-' + obj.replace('/', '-'), 'history-v4')
        receipts[obj] = result['receipt']
        atomic_json(saved_path, receipts)
        doc = receipt_data(result['receipt'])
    versions = [v for v in doc['elements'] if v['timestamp'] <= CUTOFF]
    if not versions:
        raise ValueError('no_record_by_cutoff')
    version = max(versions, key=lambda v: v['version'])
    if not version.get('visible', True):
        raise ValueError('deleted_at_cutoff')
    memo[obj] = version
    return version
def geometry_at_cutoff(obj):
    v = historical(obj)
    if obj.startswith('node/'):
        return Point(v['lon'], v['lat']), 'osm_node', v
    ids = v.get('nodes', [])
    if len(ids) < 4 or ids[0] != ids[-1]:
        raise ValueError('not_closed_facility_polygon')
    nodes = [historical('node/' + str(i)) for i in ids]
    polygon = Polygon([(n['lon'], n['lat']) for n in nodes])
    if not polygon.is_valid or polygon.is_empty:
        raise ValueError('invalid_historical_polygon')
    return polygon, 'polygon_representative', v
def address_support(address, tags):
    street = tags.get('addr:street', '')
    number = tags.get('addr:housenumber', '')
    clean = lambda s: re.sub(r'[^a-z0-9]', '', re.sub(r'\b(jalan|jl|jln|no|nomor)\b', '', str(s).casefold()))
    return bool(street and number and clean(street) in clean(address)
                and str(number) in re.findall(r'\d+', address))
records = []
for facility, candidates in discovery:
    base = dict(facility_code=facility['facility_code'], name=facility['attributes']['nama_puskesmas'],
                region_code=facility['region_code'], registered_address=facility['attributes']['alamat'],
                location_verified=False, public_eligible=False)
    if not candidates:
        records.append(dict(base, status='no_candidate', reason='Tidak ditemukan pada snapshot sumber', lon=None, lat=None))
    for obj, feature in candidates.items():
        row = dict(base, osm_object=obj, evidence_url='https://www.openstreetmap.org/' + obj,
                   candidate_count=len(candidates), status='held', lon=None, lat=None)
        try:
            geom, precision, version = geometry_at_cutoff(obj)
            tags = version.get('tags', {})
            if normalized(tags.get('name', '')) != normalized(base['name']) or excluded_name(tags.get('name', '')):
                raise ValueError('historical_name_mismatch')
            if not (tags.get('amenity') in {'hospital', 'clinic', 'doctors'} or tags.get('healthcare') in {'hospital', 'clinic', 'centre', 'doctor'}):
                raise ValueError('historical_facility_type_unclear')
            if tags.get('place'):
                raise ValueError('facility_and_place_tags_conflict')
            if not boundaries[base['region_code']].covers(geom):
                raise ValueError('outside_or_crossing_assigned_boundary')
            point = geom.representative_point()
            if not all(math.isfinite(n) for n in (point.x, point.y)):
                raise ValueError('invalid_coordinate')
            supported = address_support(base['registered_address'], tags)
            row.update(lon=point.x, lat=point.y, precision=precision,
                       osm_version=version['version'], osm_timestamp=version['timestamp'],
                       osm_tags=tags, address_supported=supported,
                       status='historical_candidate', reason='Identity requires independent official evidence')
            if len(candidates) > 1:
                row.update(status='held', reason='multiple_objects_for_facility')
        except Exception as error:
            row['reason'] = str(error)
        records.append(row)
    atomic_json(OUT / 'checkpoint.json', {'cutoff': CUTOFF, 'records': records})
groups = defaultdict(set)
for row in records:
    if row.get('lon') is not None:
        groups[(round(row['lon'], 5), round(row['lat'], 5))].add(row['facility_code'])
for row in records:
    if row.get('lon') is not None and len(groups[(round(row['lon'], 5), round(row['lat'], 5))]) > 1:
        row.update(status='held', reason='coordinate_shared_by_facilities')
print(dict(Counter(r['status'] for r in records)))

{'no_candidate': 367, 'held': 21, 'historical_candidate': 21}


## 3. Hasil dan bukti audit
`candidates.json` menyimpan seluruh hasil dan hash input. `candidate-points.geojson` hanya berisi kandidat yang lolos pemeriksaan teknis, semuanya **belum terverifikasi**. `summary.json` membedakan jumlah fasilitas dan jumlah kandidat. Tidak ada penggabungan otomatis ke lokasi produksi.

Langkah verifikasi substantif: cocokkan alamat/nomor/identitas dengan dokumen resmi independen sesuai periode, periksa konflik dan perpindahan lokasi, kemudian gunakan alur review proyek. Kecocokan alamat otomatis hanya petunjuk; tidak menggantikan langkah tersebut.

In [4]:
assert normalized('Manding') != normalized('Ganding')
assert normalized('UPTD Puskesmas Kalidawir') == normalized('KALIDAWIR')
assert excluded_name('Puskesmas Pembantu Kalidawir')
assert all(not r['location_verified'] and not r['public_eligible'] for r in records)
assert len({r['facility_code'] for r in records}) == len(targets)
assert all(r['facility_code'] not in accepted for r in records)
summary = dict(target_facilities=len(targets), candidate_records=len(records),
               facilities_with_candidates=sum(bool(c) for _, c in discovery),
               status_counts=dict(Counter(r['status'] for r in records)),
               new_verified_locations=0, database_modified=False,
               cutoff=CUTOFF, network_enabled=ALLOW_NETWORK, fetch_attempts=fetches,
               generated_at=datetime.now(timezone.utc).isoformat())
atomic_json(OUT / 'candidates.json', {'summary': summary, 'input_sha256': inputs, 'records': records})
atomic_json(OUT / 'summary.json', summary)
atomic_json(OUT / 'candidate-points.geojson', {'type': 'FeatureCollection', 'features': [
    {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
     'properties': r} for r in records if r['status'] == 'historical_candidate']})
print(json.dumps(summary, indent=2))
display(pd.DataFrame(records)[['facility_code', 'name', 'status', 'lon', 'lat', 'reason']].head(20))
print('Output:', OUT)

{
  "target_facilities": 407,
  "candidate_records": 409,
  "facilities_with_candidates": 40,
  "status_counts": {
    "no_candidate": 367,
    "held": 21,
    "historical_candidate": 21
  },
  "new_verified_locations": 0,
  "database_modified": false,
  "cutoff": "2024-12-31T23:59:59Z",
  "network_enabled": false,
  "fetch_attempts": 0,
  "generated_at": "2026-11-05T02:57:58.528561+00:00"
}


,facility_code,name,status,lon,lat,reason
0,35010200001,DONOROJO,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
1,35010200002,KALAK,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
2,35010200004,GONDOSARI,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
3,35010200005,PRINGKUKU,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
4,35010200006,CANDI,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
5,35010200010,KETROWONOJOYO,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
6,35010200012,KEDUNGBENDO,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
7,35010200013,NAWANGAN,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
8,35010200014,PAKISBARU,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber
9,35010200016,JERUK,no_candidate,NaN,NaN,Tidak ditemukan pada snapshot sumber


Output: D:\laragon\www\GIS-puskesmas\data\exports\coordinate-review-v4
